In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from xgboost import XGBClassifier

In [2]:
df=pd.read_csv("loan_approval_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'loan_approval_data.csv'

In [ ]:
df["Loan_Approved"].value_counts()

In [ ]:
# Drop ID column
df = df.drop("Applicant_ID", axis=1)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns
numerical_cols = df.select_dtypes(include=["number"]).columns

# Numerical imputation
num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

# Categorical imputation
cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

In [ ]:
classes_count = df["Loan_Approved"].value_counts()

plt.pie(classes_count, labels=["No","Yes"], autopct="%1.1f%%")
plt.title("Loan Approval Distribution")
plt.show()

In [ ]:
le = LabelEncoder()
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])

In [ ]:
X = df.drop("Loan_Approved", axis=1)
y = df["Loan_Approved"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", xgb_model)
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nAccuracy:", accuracy_score(y_test, y_pred))

In [ ]:
import joblib

joblib.dump(pipeline, "loan_model.pkl")

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
print(X.columns)

In [ ]:
print(df["Loan_Purpose"].unique())

In [ ]:
import joblib
import pandas as pd

# Load model
model = joblib.load("loan_model.pkl")

# Extract XGBoost model
xgb_model = model.named_steps['model']

# Extract preprocessor
preprocessor = model.named_steps['preprocessing']

# Get feature names after encoding
feature_names = preprocessor.get_feature_names_out()

# Get importance values
importance = pd.Series(xgb_model.feature_importances_, index=feature_names)

# Sort and display
importance_sorted = importance.sort_values(ascending=False)

print(importance_sorted)

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
df["Loan_Approved"].value_counts()

In [ ]:
df["DTI_Ratio"].describe()